<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


<h1>实验：基于 HOG 和 SVM 的图像分类</h1>



预计所需时间：**60** 分钟


## 概述


你将学习如何使用图像作为输入数据来训练支持向量机（SVM）模型。
SVM 是一种监督学习算法，常用于分类和回归任务。它通过寻找最优超平面，以最大间隔将不同类别的数据点分开。在本实验中，我们将使用 SVM 根据图像的视觉特征对图像进行分类。


## 目标


你将使用方向梯度直方图（HOG）进行特征提取，并使用支持向量机（SVM）进行分类。这种传统的计算机视觉方法在深度学习兴起之前被广泛用于图像分类任务，并且对于理解经典机器学习流程仍然很有价值。


# 目录


本笔记本分为以下几个部分：
        
- [安装与导入库](#Install-and-Import-Libraries)
- [图像处理](#Image-Processing)
- [加载图像标注](#Load-the-Image-Annotations)
- [H.O.G. 作为特征描述子](#H.O.G.-as-a-feature-descriptor) 
- [加载图像并生成训练/测试数据集](#Load-Images-and-Generate-Training/Testing-Dataset)
- [用于图像分类的 SVM](#SVM-for-Image-classification) 
- [练习](#Practice-Exercise)
        


----


# 安装与导入库


In [ ]:
!pip install --upgrade opencv-python-headless --quiet
!pip install numpy pandas matplotlib seaborn imutils scikit-learn scikit-image joblib --quiet

**导入重要库并定义辅助函数**


用于数据处理和可视化的库：


In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from imutils import paths
import seaborn as sns
import random
import time
from datetime import datetime
import requests
import zipfile
import json
import random

用于图像预处理和分类的库：


In [ ]:
import cv2
import joblib
from skimage.feature import hog
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

用于操作系统的库：


In [ ]:
import os
import io

## 图像处理


我们将加载并处理每张图像。先了解一些概念：

<ul>
        <ul>
            <li><code>cv2.resize()</code> 用于调整图像大小 </li>
            <li><code>cv2.COLOR_BGR2GRAY()</code> 将图像转换为灰度图像</li>
            <li><code>hog()</code> 从图像中提取 H.O.G. 特征 </li>
        </ul>
    
</ul>

我们将使用 **load_image()** 函数读取并预处理图像。该函数将在 **方向梯度直方图（H.O.G.）** 部分被调用。
它执行以下步骤：
<ul>
    <li>调整图像大小</li>
将每张图像标准化为 64×64 像素，以确保统一的输入尺寸。

<li>转换为灰度图</li>
将图像转换为灰度图，因为 HOG 基于强度梯度运行，不需要颜色信息。

<li>提取 HOG 特征</li>
使用 hog() 函数从灰度图像中提取方向梯度直方图特征。

<li>使用标注获取标签</li>
提取图像文件名。
在标注 JSON 文件中查找对应的标签。

<li>将标签转换为数字索引</li>
将文本标签（例如 'cat'、'dog'）转换为其在 class_object 列表中的数字索引。

<li>追加特征和标签</li>
将提取的 HOG 特征和对应的数字标签存储在 train_images 和 train_labels 列表中。
</ul>


In [ ]:
def load_images(image_paths):
    # loop over the input images with a progress bar
    for image_path in tqdm(image_paths, desc="Loading and processing images"):
        # read image
        image = cv2.imread(image_path)
        if image is None:
            tqdm.write(f"Could not read {image_path}")
            continue

        image = np.array(image).astype('uint8')
        image = cv2.resize(image, (64, 64))
        grey_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # extract HOG features
        hog_features, hog_images = hog(grey_image,
                                       visualize=True,
                                       block_norm='L2-Hys',
                                       pixels_per_cell=(16, 16))

        # get label using the filename
        filename = os.path.basename(image_path)

        if filename not in annotations["annotations"]:
            tqdm.write(f"Skipping {filename}, not found in annotations.")
            continue

        label = class_object.index(annotations["annotations"][filename][0]['label'])

        train_images.append(hog_features)
        train_labels.append(label)

### 下载图像和标注
我们将使用 <code>Sklearn</code> 库中的 SVM 分类器来训练和分类这些图像。开始之前，先获取图像并查看其中一些。


In [ ]:
# URL of the ZIP file
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QMmv2Y3pAQe39dV4t0ZJrA/training-an-image-classifier-w-2025-05-22-t-09-51-08-896-z.zip"

# Send a GET request to the URL
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Open the zip file from the downloaded content
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        zip_ref.extractall("cats_dogs")  # Extract to a target folder
    print("Download and extraction complete.")
else:
    print("Failed to download file:", response.status_code)


在图像分类中，标注是描述图像内容的标签或元数据。这些标签对于训练监督式机器学习模型至关重要。标注将以 JSON 文件的形式提供。你可以看到图像名作为键，dog 或 cat 作为标签对象。

让我们查看刚下载的标注格式。以下代码将只显示前 5 条标注。


In [ ]:
# Define the path to the annotations JSON file
annotations_path = "cats_dogs/training-an-image-classifier-w-2025-05-22-t-09-51-08-896-z/_annotations.json"

# Load the JSON file
with open(annotations_path, "r") as f:
    annotations = json.load(f)

# Now safely access the first five entries
first_five = {k: annotations["annotations"][k] for k in list(annotations["annotations"])[:5]}
first_five


# 加载图像标注


In [ ]:
# Define base folder path
base_folder = "cats_dogs/training-an-image-classifier-w-2025-05-22-t-09-51-08-896-z"

# Path to the annotations JSON file
annotations_path = os.path.join(base_folder, "_annotations.json")

# Load the JSON data
with open(annotations_path, "r") as f:
    annotations = json.load(f)

print("Annotations loaded successfully!")

## H.O.G. 作为特征描述子


H.O.G. 为每个局部区域生成一个直方图。我们将随机选择一张图像，看看 H.O.G. 是如何工作的。


In [ ]:
# Pick a random image from the annotations
random_image_name = random.choice(list(annotations["annotations"].keys()))

# Get the label for that image
label = annotations["annotations"][random_image_name][0]["label"]

print(f"Random image selected: {random_image_name}")
print(f"Label: {label}")

构建图像的完整路径


In [ ]:
# Construct full path to image
image_path = os.path.join(base_folder, random_image_name)

# Read image using OpenCV
img = cv2.imread(image_path)
if img is None:
    raise FileNotFoundError(f"Image not found at {image_path}")
print("Full image path:", image_path)


为了创建 H.O.G. 特征，我们首先将图像转换为灰度图像。


In [ ]:
sample_image = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
plt.figure(figsize=(10,10))
plt.imshow(sample_image, cmap = "gray")
plt.show()

将图像调整为较小尺寸以加快算法运行，并将图像转换为灰度图以减少通道数。`OpenCV` 以 `BGR` 格式读取图像，因此我们将使用该颜色通道转换为灰度。

`OpenCV` 的早期开发者选择 `BGR` 颜色格式，因为当时该格式在相机制造商和软件供应商中很流行。


In [ ]:
sample_image = cv2.resize(img, (64, 64))
sample_image = cv2.cvtColor(sample_image, cv2.COLOR_BGR2GRAY)

绘制数据，看看它是什么样子：


In [ ]:
plt.imshow(sample_image, cmap=plt.cm.gray)

在灰度图像上运行 H.O.G.，看看效果。

H.O.G. 是方向梯度直方图（Histogram of Oriented Gradients）的缩写。它利用图像局部区域的梯度方向，并为每个局部区域生成一个直方图。


In [ ]:
## when we run H.O.G., it returns an array of features and the image/output it produced
## the featurre is what we use to train the SVM model
sample_image_features, sample_hog_image = hog(sample_image,
                              visualize=True,
                              block_norm='L2-Hys',
                              pixels_per_cell=(16, 16))

## lets look at what the H.O.G. feature looks like
plt.imshow(sample_hog_image, cmap=plt.cm.gray)

## 加载图像并生成训练/测试数据集


初始化保存已加载图像的位置：


In [ ]:
image_paths = list(paths.list_images(base_folder))
train_images = []
train_labels = []
class_object = annotations['labels']

在图像路径上使用该函数：


In [ ]:
load_images(image_paths)

In [ ]:
print(f"Number of images: {len(train_images)}")
print(f"Number of labels: {len(train_labels)}")

创建图像数组，并使用 <code>np.vstack</code> 垂直堆叠数组以便处理。


In [ ]:
train_array = np.array(train_images)
train_array = np.vstack(train_array)

我们将把数组 <code>reshape</code> 为 <code>(标签大小, 1)</code>。数组将如下所示：<code>[[1], [0], ..., [0]]</code></p>


In [ ]:
labels_array = np.array(train_labels)

In [ ]:
labels_array = labels_array.astype(int)
labels_array = labels_array.reshape((labels_array.size,1))

拼接图像和标签：


In [ ]:
train_df = np.concatenate([train_array, labels_array], axis = 1)

将数据划分为训练集和测试集：


In [ ]:
percentage = 70
partition = int(len(train_df)*percentage/100)

In [ ]:
x_train, x_test = train_df[:partition,:-1],  train_df[partition:,:-1]
y_train, y_test = train_df[:partition,-1:].ravel(), train_df[partition:,-1:].ravel()

### 超参数


要使用的核函数类型是一种超参数。最常见的核函数有 <code>RBF</code>、<code>poly</code> 或 <code>sigmoid</code>。你也可以创建自己的核函数。

<code>C</code> 在 SVM 中充当正则化参数。<code>C</code> 参数在训练样本的正确分类与决策函数间隔最大化之间进行权衡。<code>C</code> 值越大，如果决策函数能更好地正确分类所有训练点，则接受更小的间隔；<code>C</code> 值越低，则鼓励更大的间隔，因此得到更简单的决策函数，但会牺牲准确率。我们使用验证数据来选择 C 和最佳核函数。


Python 字典 <code>param_grid</code> 包含不同的核函数和 C 值。我们可以使用验证数据来测试它们。


In [ ]:
param_grid = {'kernel': ('linear', 'rbf'),'C': [1, 10, 100]}

<code>gamma</code> 是 RBF 核函数的参数，可以看作核函数的扩散范围，也就是决策区域。低值表示“远”，高值表示“近”。模型行为对 gamma 参数非常敏感。如果 gamma 过大，支持向量的影响区域半径将只包含支持向量本身。我们创建一个支持向量分类对象。


## 用于图像分类的 SVM


In [ ]:
base_estimator = SVC(gamma='scale')

我们将使用 <code>GridSearchCV</code> 函数训练模型，并尝试不同的核函数和参数值。最终输出的是在验证数据上表现最好的模型。


In [ ]:
start_datetime = datetime.now()
start = time.time()

svm = GridSearchCV(base_estimator, param_grid, cv=5)
#Fit the data into the classifier
svm.fit(x_train,y_train)
#Get values of the grid search
best_parameters = svm.best_params_
print(best_parameters)
#Predict on the validation set
y_pred = svm.predict(x_test)
# Print accuracy score for the model on validation  set. 
print("Accuracy: "+str(accuracy_score(y_test, y_pred)))

end = time.time()
end_datetime = datetime.now()
print(end - start)

**混淆矩阵快速指南**


混淆矩阵是分类问题的性能度量工具。它是一个预测值与实际值组合的表格。y 轴是 `真实` 标签，x 轴是 `预测` 标签。本示例关注二分类器，即“是/否”模型。

<table>
  <tr>
    <td>&nbsp;</td>
    <td>预测：否</td>
    <td>预测：是</td>
  </tr>
  <tr>
    <td>真实：否</td>
    <td>30</td>
    <td>30</td>
  </tr>
  <tr>
    <td>真实：是</td>
    <td>10</td>
    <td>50</td>
  </tr>
</table>

在这个矩阵中，我们可以看到有两个类别。例如，如果我们在预测一张图像是否热狗，“是”表示它是热狗，“否”表示它不是热狗。我们有 120 次预测，其中分类器预测“是”80 次，“否”40 次，但实际上有 60 个“是”和 60 个“否”。

谈到混淆矩阵时，我们涉及几个术语：
* 真正例（TP）：模型预测为“是”，实际也是“是”
* 真负例（TN）：模型预测为“否”，实际也是“否”
* 假正例（FP）：模型预测为“是”，但实际是“否”
* 假负例（FN）：模型预测为“否”，但实际是“是”

结合我们的示例来看：

<table>
  <tr>
    <td>&nbsp;</td>
    <td>预测：否</td>
    <td>预测：是</td>
  </tr>
  <tr>
    <td>真实：否</td>
    <td>TN = 30</td>
    <td>FP = 30</td>
    <td>60</td>
  </tr>
  <tr>
    <td>真实：是</td>
    <td>FN = 10</td>
    <td>TP = 50</td>
    <td>60</td>
  </tr>
  <tr>
    <td>&nbsp;</td>
    <td>40</td>
    <td>80</td>
  </tr>
</table>

**准确率** 是模型预测正确的数量除以总预测数量。即 (TP+TN)/总预测数量。


获取 SVM 结果的混淆矩阵：


In [ ]:
label_names = [0, 1]
cmx = confusion_matrix(y_test, y_pred, labels=label_names)

In [ ]:
# Convert the confusion matrix array into a pandas DataFrame for easier plotting
df_cm = pd.DataFrame(cmx)

# Optional: set font scale for Seaborn heatmap labels (e.g., axis labels and annotations)
sns.set(font_scale=1.4)  # Increases the font size for better readability

# Plot the confusion matrix using a heatmap
# - annot=True enables showing the numeric values inside the cells
# fmt="d" forces integer display
# - annot_kws controls the annotation text size
sns.heatmap(df_cm, annot=True, fmt="d",annot_kws={"size": 16})

# Set the plot title to indicate this is the confusion matrix for SVM classification
title = "Confusion Matrix for SVM results"
plt.title(title)

# Display the plot
plt.show()


In [ ]:
# 将 SVM 模型保存到文件
joblib.dump(svm.best_estimator_, 'svm.joblib')

# 练习
### 使用上传的图像测试模型


上传你的图像，看看是否能被正确分类。
<p><b>如何上传图像：</b></p>
使用上传按钮从本地机器上传图像：
<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-SkillsNetwork/images/instruction.png" width="300"  />
</center>


图像现在会出现在你当前工作的目录中。要在新单元格中读取图像，请使用 <code>cv2.imread</code> 并指定其名称。例如，我将 <code>anothercar.jpg</code> 上传到当前工作目录——<code>cv2.imread("anothercar.jpg")</code>。

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-SkillsNetwork/images/instruction2.png" width="300"  />
</center>


或者使用下面的图像进行测试。


In [ ]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/A3xVgxTJVdrZTtBwGVbEIw/Cat.jpg"
!wget"https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/KZnFQiZj3e_sQKIyfvHvTA/dog.jpg"

加载模型


In [ ]:
svm_model = joblib.load('svm.joblib')

将下面的 your_uploaded_file 替换为你目录中显示的图像名称。如果你使用的是笔记本中提供的下载图像，则使用 `Cat.jpg` 或 `dog.jpg`


In [ ]:
my_image = cv2.imread("test1.png")
## let's see what the image looks like
image = cv2.cvtColor(my_image, cv2.COLOR_BGR2RGB)
plt.imshow(image)
plt.axis('off')
plt.show()

图像预处理与 HOG 特征提取


In [ ]:

# Check if image is loaded
if image is None:
    raise ValueError("Image not found or path is incorrect.")

# Resize and convert to grayscale
image = cv2.resize(image, (64, 64))
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Extract HOG features
hog_features, hog_image = hog(gray,
                              visualize=True,
                              block_norm='L2-Hys',
                              pixels_per_cell=(16, 16))

# Convert to appropriate shape for prediction
hog_features = np.array([hog_features])  # Shape: (1, N)


使用预训练 SVM 模型进行图像分类


In [ ]:
# Predict using the loaded model
predicted_label_index = svm_model.predict(hog_features)[0]

# Convert to integer for list indexing
predicted_label = annotations["labels"][int(predicted_label_index)]

print(f"The image was classified as: {predicted_label}")


显示 HOG 图像


In [ ]:
# Display the HOG image
plt.figure(figsize=(6, 6))
plt.imshow(hog_image, cmap='gray')
plt.title(f"HOG Visualization - Predicted: {predicted_label}")
plt.axis("off")
plt.show()

该可视化展示了模型如何使用 HOG（方向梯度直方图）特征来解释图像以进行分类。

黑白单元格的网格代表图像不同区域的局部梯度信息。

每个单元格中明亮的线条或方向表示该部分图像中检测到的边缘方向和强度。

区域越亮，表示该方向上检测到的边缘或梯度越强。


## 恭喜！

你已成功完成了使用支持向量机（SVM）算法和 OpenCV 的完整图像分类流程！


<h2>作者</h2>


 [Aije Egwaikhide](https://www.linkedin.com/in/aije-egwaikhide/)

 [Sathya Priya](https://www.linkedin.com/in/sathya-priya-06120a17a/) 


<!--<h2>Change Log</h2>-->


<!--<table>
    <tr>
        <th>Date (YYYY-MM-DD)</th>
        <th>Version</th>
        <th>Changed By</th>
        <th>Change Description</th>
    </tr>
    </tr>
        <tr>
        <td>2025-06-25</td>
        <td>1.3</td>
        <td>Sathya</td>
        <td>Created and Converted the lab to JupyterCurrent notebook</td>
    </tr>
    <tr>
        <td>2021-05-25</td>
        <td>1.2</td>
        <td>Kathy</td>
        <td>Modified multiple areas</td>
    </tr>
    <tr>
        <td>2021-05-25</td>
        <td>1.2</td>
        <td>Yasmine</td>
        <td>Modified multiple areas</td>
    </tr>
     <tr>
        <td>2021-04-10</td>
        <td>1.1</td>
        <td>Aije</td>
        <td>Fixed grammatical errors</td>
    </tr>
    <tr>
        <td>2021-04-09</td>
        <td>1.0</td>
        <td>Aije</td>
        <td>Updated to new template</td>
    </tr>
    <tr>
        <td>2021-02-24</td>
        <td>0.1</td>
        <td>Aije</td>
        <td>Created original version of the lab</td>
    </tr>
</table>-->


<h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
